# Laboratório PySpark: A Cura para o Caos (Otimização)

Neste notebook, pegaremos o mesmo problema pesado e aplicaremos 3 conceitos clássicos de otimização de Big Data exigidos em qualquer entrevista de Engenharia de Dados Sênior:
1. **Schema Explícito & Escrita Colunar (Parquet)**: Acaba com o *Small Files Problem*.
2. **Técnica de Salting**: O antídoto definitivo para *Data Skew* em Agrupamentos (GroupBy).
3. **Broadcast Join**: Como fazer Joins sem explodir a memória (eliminando o Shuffle).

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.sql.functions import col, count, rand, lit, concat_ws, broadcast

spark = SparkSession.builder \
    .appName("Lab: Optimized Lakehouse") \
    .getOrCreate()

spark

### 1. Curando o Small Files Problem e InferSchema (Camada Silver)
Em vez de deixar o Spark sofrer para abrir 2000 arquivos e adivinhar os tipos de dados (`inferSchema=True`), nós definimos o Schema no código e lemos tudo de uma vez. Depois, compactamos e escrevemos o resultado em **apenas 4 arquivos no formato colunar Parquet**, simulando a transição da camada Landing para a camada Silver do Data Lake.

In [2]:
raw_path = "/home/jovyan/work/data/landing/churn_data_skewed/"
silver_path = "/home/jovyan/work/data/silver/churn_optimized.parquet"

# Dica de Ouro: Sempre force o Schema na leitura de CSVs gigantes.
schema = StructType([
    StructField("customerID", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("SeniorCitizen", IntegerType(), True),
    StructField("MonthlyCharges", DoubleType(), True),
    StructField("TotalCharges", StringType(), True)
])

print("Lendo rápido com Schema explícito...")
df_raw = spark.read.schema(schema).csv(raw_path, header=True)

print("Compactando 2000 arquivos CSV em apenas 4 arquivos Parquet otimizados...")
# O coalesce(4) junta as 2000 partições minúsculas em apenas 4 blocos robustos na memória antes de escrever no disco.
df_raw.coalesce(4).write.mode("overwrite").parquet(silver_path)
print("Sucesso! Dados salvos na camada Silver em Parquet.")

Lendo rápido com Schema explícito...
Compactando 2000 arquivos CSV em apenas 4 arquivos Parquet otimizados...
Sucesso! Dados salvos na camada Silver em Parquet.


### 2. A Magia do SALTING (A cura para o Data Skew no GroupBy)
Se nós usarmos `groupBy("customerID")` agora, a chave `SK-999999-SKEW` (que tem 2.8 milhões de linhas) voltará a sobrecarregar 1 único núcleo da CPU (o Skew).

Para resolver isso, nós aplicamos o **Salting**: 
1. Injetamos um número aleatório (ex: de 0 a 9) no final de todas as chaves.
2. Fazemos um primeiro agrupamento. Isso espalha perfeitamente os 2.8M registros entre 10 CPUs diferentes!
3. Retiramos o "sal" e reagrupamos. Como o volume de dados já caiu 99%, essa etapa é instantânea.

In [3]:
# Lendo os dados já otimizados em Parquet (MUITO mais rápido que CSV)
df_silver = spark.read.parquet(silver_path)

# PASSO 1 DO SALTING: Injetar um "sal" aleatório de 0 a 9 na chave
salt_bins = 10
df_salted = df_silver.withColumn("salt", (rand() * salt_bins).cast("int"))
df_salted = df_salted.withColumn("salted_customerID", concat_ws("_", col("customerID"), col("salt")))

# PASSO 2: Primeiro GroupBy na chave salgada (Isso espalha o trabalho perfeitamente entre os cores)
df_partial_agg = df_salted.groupBy("salted_customerID", "customerID").agg(count("*").alias("partial_calls"))

# PASSO 3: Segundo GroupBy (reagrupando a chave original, consolidando os 10 pedaços da CPU de volta num só).
df_final_agg = df_partial_agg.groupBy("customerID").agg({'partial_calls': 'sum'}).withColumnRenamed("sum(partial_calls)", "total_calls")

print("Otimização com Salting rodando... Verifique a Spark UI e veja que NENHUMA Task vai demorar mais que as outras!")
df_final_agg.orderBy("total_calls", ascending=False).show(10, truncate=False)

Otimização com Salting rodando... Verifique a Spark UI e veja que NENHUMA Task vai demorar mais que as outras!
+----------+-----------+
|customerID|total_calls|
+----------+-----------+
|Male      |1777500    |
|Female    |1744000    |
+----------+-----------+



### 3. Otimização do Join com Broadcast (A cura para o Join da Morte)
O *Join da Morte* explodiu a memória (OOM) porque tentou embaralhar e combinar milhões de registros da mesma chave pela rede (Shuffle).

Sempre que você precisar cruzar um dataset GIGANTE com um dataset PEQUENO (ex: cruzando as ligações com uma tabela de dimensões de DDD/Estado), você deve forçar um **Broadcast**.
O Broadcast copia a tabela pequena inteira para a memória local de CADA Executor do cluster. Isso elimina completamente a fase de rede (Shuffle) e o Join ocorre nativamente na memória!

In [4]:
# Simulando uma tabela auxiliar bem pequena (ex: buscando apenas os VIPs da base)
df_small_vips = df_silver.filter(col("SeniorCitizen") == 1).select("customerID").dropDuplicates()

print("Iniciando o Join Otimizado com Broadcast...")
# O comando `broadcast()` "engana" o Spark e força a cópia da tabela para a memória, evitando o Shuffle Cartesiano.
df_optimized_join = df_silver.join(broadcast(df_small_vips), on="customerID", how="inner")

# Essa contagem será extremamente rápida!
print(f"Total do cruzamento otimizado: {df_optimized_join.count()}")

Iniciando o Join Otimizado com Broadcast...
Total do cruzamento otimizado: 0
